# Food Waste Level Prediction - walkthrough

Run each cell top to bottom with **Shift + Enter**. Kernel: **Python (food-waste venv)**.

This notebook uses the same code as the production pipeline (the `foodwaste` package),
so what you see here is exactly what CI, the API and Docker run.

## Step 1 - Setup

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import pandas as pd
import matplotlib.pyplot as plt
from foodwaste import config
from foodwaste.data import load_data, validate, add_features, DataValidationError
print("Project root:", ROOT)

## Step 2 - Load and explore the data

In [ ]:
df = load_data()
print(df.shape)
df.head()

In [ ]:
print("Target distribution:")
print(df[config.TARGET].value_counts(normalize=True).round(3))
print()
print("Missing values per column:")
print(df.isna().sum()[df.isna().sum() > 0])

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
pd.crosstab(df["Food_Category"], df[config.TARGET], normalize="index")[config.CLASSES].plot.bar(
    stacked=True, ax=axes[0], rot=30, title="Waste level by food category")
df.assign(ratio=df["Purchase_Quantity"] / df["Daily_Demand"]).boxplot(
    column="ratio", by=config.TARGET, ax=axes[1], showfliers=False)
axes[1].set_title("Order / demand ratio by waste level"); plt.suptitle("")
plt.tight_layout(); plt.show()

## Step 3 - Avoid data leakage
These columns are only known **after** the day ends, so the model must not use them.
Watch what happens if we cheat and use `Waste_Quantity`:

In [ ]:
print("Leakage columns:", config.LEAKAGE_COLUMNS)
print("Features used:  ", config.RAW_FEATURES)

from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import cross_val_score
leaky_x = (df["Waste_Quantity"] / df["Purchase_Quantity"]).to_frame()
print()
print("Accuracy using the leaked column:", cross_val_score(DecisionTreeClassifier(max_depth=4), leaky_x, df[config.TARGET]).mean().round(3))
print("Looks amazing - but useless in real life, because you don't know the waste until the day is over.")

## Step 4 - Data validation (what stops bad data reaching the model)

In [ ]:
validate(df)
print("Real data: OK")

bad = df.head(5).copy()
bad.loc[0, "Food_Category"] = "Seafood"
bad.loc[1, "Purchase_Quantity"] = -10
try:
    validate(bad)
except DataValidationError as e:
    print("Bad data rejected:", e)

## Step 5 - Feature engineering

In [ ]:
add_features(df[config.RAW_FEATURES].head())

## Step 6 - Train, compare models and apply the quality gate

In [ ]:
from foodwaste.train import train
model, metrics, passed = train()

In [ ]:
pd.DataFrame(metrics["candidates"]).T

## Step 7 - Confusion matrix of the best model

In [ ]:
from IPython.display import Image
Image(str(config.MODELS_DIR / "confusion_matrix.png"))

## Step 8 - Predict for a new day

In [ ]:
from foodwaste.predict import predict, load_model
load_model.cache_clear()

today = {"Date": "2025-07-14", "Business_Type": "Restaurant", "Food_Category": "Prepared Food",
         "Purchase_Quantity": 180, "Shelf_Life_Days": 1, "Storage_Temperature": 24.0,
         "Daily_Demand": 110, "Promotion": "No", "Weather": "Rainy"}
print("Over-ordered:  ", predict([today])[0])
print("Right-sized:   ", predict([{**today, "Purchase_Quantity": 110}])[0])

## Step 9 - Drift monitoring
Compare new data with the training data. A normal batch should pass; a simulated heatwave should be flagged.

In [ ]:
from foodwaste.drift import detect_drift, simulate
reference = pd.read_csv(config.REFERENCE_PATH)

print("NORMAL BATCH")
display(detect_drift(reference, simulate("normal"))[["feature", "statistic", "p_value", "drifted"]])
print("HEATWAVE BATCH")
display(detect_drift(reference, simulate("heatwave"))[["feature", "statistic", "p_value", "drifted"]])

## Next
In a terminal (venv activated), from the project root:
```
pytest                                   # run all tests
uvicorn api.main:app --reload            # API  -> http://localhost:8000/docs
streamlit run app/streamlit_app.py       # UI   -> http://localhost:8501
python -m foodwaste.drift --simulate heatwave   # HTML report in reports/
```